# 02 — Data Preprocessing

Inspect data/class distributions, image quality and the original operator-calibration evidence, then apply the frozen v2 Gamma-or-Skip policy to matched training and development inputs. Research/EDA appears below as readable sections; no detector training or inference runs.

- **Inputs:** Frozen phase-01 protocol, original source images, converted development labels/Raw references, frozen v2 policy and calibration-screen evidence.
- **Outputs:** Default: `runs/preprocessing-five-v1/` five-class labels, matched exports and annotation-only support. With `FIVE_CLASS_ONLY=False`, also `runs/preprocessing-v1/`: nested S10/S20 manifests, matched S10 and development inputs, frozen recipe; `docs/preprocessing-v1/`: quality/cost evidence.
- **Prerequisites:** Study environment, declared phase-01 artifacts, committed calibration evidence, and sufficient export disk space.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Dataset and Class Distributions](#dataset-distributions)
3. [Quality Diagnostics and Operator Evidence](#quality-operator-evidence)
4. [Paired Training Export](#paired-training-export)
5. [Paired Quality and Offline Costs](#paired-quality-costs)
6. [Development Inputs and Synthetic Evidence](#development-synthetic-evidence)
7. [Export and Integrity Summary](#export-integrity)
8. [Derived Five-Class Inputs and Summary](#five-class-preparation)

Set paths in section 1, then **Restart & Run All**. The default `FIVE_CLASS_ONLY=True` reads saved six-class subsets/conversions and
skips historical sections 2–7. Set it to `False` only to prepare a fresh six-class
phase before deriving five-class inputs. Only this phase executes.
Missing or mismatched required artifacts fail; upstream phases are never launched.
The historical six-class taxonomy, including railway `train`, stays unchanged.


<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

Configure artifact paths. The policy is a previously selected calibration artifact, never fitted from development or final-test outcomes.

In [ ]:
import gzip
import os
import platform
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

from vehicle_detection.adaptive import (
    METRICS,
    export_pair,
    paired_quality,
    validate_policy,
)
from vehicle_detection.common import (
    file_hash,
    freeze_json,
    read_image,
    read_json,
    require_paths,
    write_json,
)
from vehicle_detection.data import CLASSES, convert_labels, scan_bdd
from vehicle_detection.datasets import export_development_pair
from vehicle_detection.offline import estimate_disk, verify_alignment
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.subsets import (
    build_subsets,
    freeze_subsets,
    training_records,
    validate_subsets,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
FIVE_CLASS_ONLY = True
FIVE_ROOT = ROOT / "runs/preprocessing-five-v1"
RUN_ROOT = ROOT / "runs/preprocessing-v1"
REPORT_ROOT = ROOT / "docs/preprocessing-v1"
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
SEED, ATTEMPTS = 2026, 128
INCLUDE_S40 = False
EXPORT_SUBSET = "S10"  # S20 is a manifest only until a later scaling decision.
FROZEN_POLICY = ROOT / "docs/issue6-v2/policy.json"
RAW_DEVELOPMENT = ROOT / "runs/data-v1/datasets/raw"
DEV_CONVERSION = ROOT / "runs/data-v1/conversion"
FROZEN_SUBSETS = RUN_ROOT / "subsets.json"
TRAIN_CONVERSION = RUN_ROOT / "conversion" / EXPORT_SUBSET
require_paths(
    protocol=PROTOCOL_PATH,
    data=DATA_ROOT,
    policy=FROZEN_POLICY,
    development_labels=DEV_CONVERSION / "conversion.json",
    decision=ROOT / "docs/issue11-v2/v3_decision.json",
    screening=ROOT / "docs/issue11-v2/experimental_matrix.csv",
    screening_figure=ROOT / "docs/issue11-v2/supported_recovery.png",
)
if FIVE_CLASS_ONLY:
    require_paths(
        subsets=FROZEN_SUBSETS,
        training_labels=TRAIN_CONVERSION / "conversion.json",
    )
else:
    require_paths(raw_development=RAW_DEVELOPMENT / "verified.json")
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
cv2.setNumThreads(1)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})
protocol = validate_protocol(read_json(PROTOCOL_PATH))
print("Frozen parent:", protocol["manifest_id"])
print("No detector training/inference or final-test pixel access in this notebook.")

<a id="dataset-distributions"></a>

## 2. Dataset and Class Distributions

Freeze representative nested whole-group subsets using training-pool class/condition drift. S20 stays a manifest until a separately justified expansion.

### 2.1 Membership, Class Support and Imbalance

In [ ]:
if not FIVE_CLASS_ONLY:
    subsets = build_subsets(protocol, seed=SEED, attempts=ATTEMPTS, include_s40=INCLUDE_S40)
    freeze_subsets(RUN_ROOT / "subsets.json", subsets, protocol)
    pool = protocol["splits"]["training_pool"]
    counts = pd.DataFrame(
        {
            "training_pool": protocol["summaries"]["training_pool"],
            **{k: v["summary"] for k, v in subsets["subsets"].items()},
        }
    ).fillna(0)
    display(counts.loc[["images", "negative", *[f"presence:{c}" for c in CLASSES]]].T)
    print("Subset identity:", subsets["manifest_id"])
    display(
        pd.DataFrame(
            [
                {
                    "subset": k,
                    "groups": len(v["groups"]),
                    "target_fraction": v["target_fraction"],
                    "actual_fraction": v["actual_fraction"],
                }
                for k, v in subsets["subsets"].items()
            ]
        )
    )
    audit = pd.DataFrame(subsets["diagnostics"])
    audit.to_csv(REPORT_ROOT / "subset_support.csv", index=False)
    display(audit[audit.rare_support].sort_values(["feature", "subset"]))
    display(audit.assign(abs_drift=audit.delta_per_image.abs()).nlargest(15, "abs_drift"))
    print("\n".join(subsets["limitations"]))

### 2.2 Conditions, Object Sizes and Backgrounds

In [ ]:
if not FIVE_CLASS_ONLY:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, prefix in zip(axes, ["presence", "instances"], strict=True):
        data = counts.reindex([f"{prefix}:{c}" for c in CLASSES], fill_value=0)
        data.index = CLASSES
        data.div(counts.loc["images"]).plot.bar(ax=ax, title=f"Class {prefix} / image", rot=20)
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "class_support.png")
    plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, axis in zip(axes, ["weather", "timeofday", "scene"], strict=True):
        data = counts.loc[[k for k in counts.index if k.startswith(axis + ":")]]
        data.index = data.index.str.split(":", n=1).str[1]
        data.div(counts.loc["images"]).plot.bar(ax=ax, title=axis, rot=65)
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "conditions.png")
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    counts.reindex([f"size_instances:{s}" for s in ["small", "medium", "large"]], fill_value=0).div(
        counts.loc["images"]
    ).plot.bar(ax=axes[0], title="Box-size histogram / image", rot=20)
    counts.loc[["negative"]].div(counts.loc["images"]).T.plot.bar(
        ax=axes[1], title="Negative image fraction", legend=False, rot=20
    )
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "sizes_negatives.png")
    plt.show()

These distributions describe natural support and imbalance; rare classes/conditions are not oversampled or removed. All six class IDs and source memberships remain fixed.

<a id="quality-operator-evidence"></a>

## 3. Quality Diagnostics and Operator Evidence

Inspect the frozen calibration-only measurements/candidates. These are original evidence, not a new calibration or detection result.

### 3.1 Image Quality and Independent Operator Ablations

In [ ]:
if not FIVE_CLASS_ONLY:
    policy = validate_policy(read_json(FROZEN_POLICY), protocol)
    freeze_json(RUN_ROOT / "policy.json", policy)
    print("Frozen policy before pilot inspection:", policy["policy_id"])
    display(policy["recipes"])
    display(
        pd.DataFrame(policy["route_support"]).T[
            ["eligible_count", "candidate_count", "minimum_support", "status"]
        ]
    )
    display(policy["noise_diagnostics"])
    display(pd.DataFrame(policy["candidates"]))
    calibration = pd.DataFrame(policy["measurements"])
    calibration[list(METRICS[:4])].hist(bins=24, figsize=(10, 5))
    plt.suptitle("Calibration sample · quality proxies only")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / "calibration_quality.png")
    plt.show()
    print("\n".join(policy["limits"]))

Gamma-or-Skip remains the evidence-selected v2 outcome. Unsupported CLAHE/Denoise/Sharpen routes stay disabled; the noise proxy remains unvalidated. Better quality proxies do not establish detector gains.

<a id="paired-training-export"></a>

## 4. Paired Training Export

Estimate disk use, convert labels once, then export exactly the matched Raw/Adaptive S10 pair. Both use identical IDs/GT, source dimensions and JPEG95; no final inputs are prepared.

In [ ]:
if not FIVE_CLASS_ONLY:
    records, _, sources = scan_bdd(DATA_ROOT, strict=False)
    for split, source in sources.items():
        assert source["sha256"] == protocol["annotation_sources"][split]["sha256"]
    partitions = training_records(protocol, subsets, EXPORT_SUBSET, records, DATA_ROOT)
    selected = partitions["train"]
    sample = [
        selected[i] for i in np.linspace(0, len(selected) - 1, min(16, len(selected)), dtype=int)
    ]
    budget = estimate_disk(sample, len(selected), ["raw_reencoded", "adaptive"], policy["recipes"])
    budget.to_csv(REPORT_ROOT / "storage_estimate.csv", index=False)
    required = float(budget.p90_budget_GiB.sum() * 1.25 + 1)
    free = shutil.disk_usage(RUN_ROOT).free / 2**30
    display(budget)
    print(f"Estimated headroom requirement {required:.2f} GiB; free {free:.2f} GiB")
    assert free >= required, "Insufficient estimated storage; do not start the export"
    converted = RUN_ROOT / "conversion" / EXPORT_SUBSET
    with capture_output() as conversion_log:
        conversion = convert_labels(partitions, converted, convention="continuous")
    (RUN_ROOT / f"{EXPORT_SUBSET}_conversion.log").write_text(
        conversion_log.stdout + conversion_log.stderr
    )
    pair_root = RUN_ROOT / "datasets" / EXPORT_SUBSET
    tables = export_pair(partitions, converted, pair_root, policy, protocol, subsets, EXPORT_SUBSET)
    assert verify_alignment([pair_root / b for b in tables]) == len(selected)
    for branch in tables:
        assert not list((pair_root / branch / "images/test").iterdir())
        assert not list((pair_root / branch / "images/tune").iterdir())
    print("Deep-verified paired images per branch:", len(selected))

<a id="paired-quality-costs"></a>

## 5. Paired Quality and Offline Costs

Quality deltas compare decoded JPEGs under equal encoding. Environment labels join only after freeze; CPU enhancement/preparation timing is separate from model latency.

### 5.1 Quality, Routing and Latency Diagnostics

In [ ]:
if not FIVE_CLASS_ONLY:
    paired = paired_quality(pair_root)
    paired.to_csv(pair_root / "paired_quality.csv", index=False)
    meta = tables["adaptive"].copy()
    meta["filters"] = meta.decisions.map(len)
    meta["decision"] = meta.decisions.map(lambda d: d[0]["name"] if d else "skip")
    assert meta.filters.max() <= 1
    attributes = pd.DataFrame([{"id": r["id"], **r["attributes"]} for r in pool])
    analysis = paired.merge(
        meta[["id", "decision", "filters", "preprocess_ms", "preparation_ms"]],
        on="id",
        validate="one_to_one",
    ).merge(attributes, on="id", validate="one_to_one")
    analysis["brightness_stratum"] = pd.cut(
        analysis.raw_brightness,
        [0, 0.2, 0.4, 1],
        labels=["dark", "middle", "bright"],
        include_lowest=True,
    )
    analysis["noise_stratum"] = np.where(
        analysis.raw_noise > 0.035, "noisy proxy", "low noise proxy"
    )
    analysis.to_csv(pair_root / "analysis.csv", index=False)
    display(analysis.decision.value_counts().rename("images"))
    print("Skip rate:", float((analysis.decision == "skip").mean()))
    print("Filter counts:", analysis.filters.value_counts().to_dict())
    latency = pd.DataFrame(
        [
            {
                "branch": b,
                "stage": stage,
                "mean_ms": t[stage].mean(),
                "p50_ms": t[stage].quantile(0.5),
                "p95_ms": t[stage].quantile(0.95),
            }
            for b, t in tables.items()
            for stage in ["preprocess_ms", "preparation_ms"]
        ]
    )
    display(latency)
    latency.to_csv(REPORT_ROOT / "offline_latency.csv", index=False)
    fig, axes = plt.subplots(2, 4, figsize=(15, 6))
    for j, metric in enumerate(METRICS[:4]):
        analysis[[f"raw_{metric}", f"adaptive_{metric}"]].plot.hist(
            ax=axes[0, j], bins=32, alpha=0.5, title=metric
        )
        analysis[f"delta_{metric}"].plot.hist(
            ax=axes[1, j], bins=32, title=f"Paired delta {metric}"
        )
        axes[1, j].axvline(0, color="black", linewidth=1)
    fig.suptitle("Image-quality proxies · no detection metrics")
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "paired_quality.png")
    plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, axis in zip(axes, ["weather", "timeofday", "scene"], strict=True):
        pd.crosstab(analysis[axis], analysis.decision, normalize="index").plot.bar(
            stacked=True, ax=ax, title=f"Decision fraction by {axis}", rot=65
        )
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "routing_environment.png")
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, axis in zip(axes, ["brightness_stratum", "noise_stratum"], strict=True):
        pd.crosstab(analysis[axis], analysis.decision, normalize="index").plot.bar(
            stacked=True, ax=ax, title=axis.replace("_", " "), rot=20
        )
    fig.tight_layout()
    fig.savefig(REPORT_ROOT / "routing_quality.png")
    plt.show()
    display(analysis.groupby("decision").proxy_loss_delta.agg(["count", "mean", "min", "max"]))

### 5.2 Representative Improvements, Skips and Adverse Cases

In [ ]:
if not FIVE_CLASS_ONLY:
    changed = analysis[analysis.decision != "skip"].sort_values(["proxy_loss_delta", "id"])
    examples = []
    if len(changed):
        examples += [
            ("Best proxy change", changed.iloc[0]["id"]),
            ("Worst proxy change / adverse candidate", changed.iloc[-1]["id"]),
        ]
        clipping = changed.sort_values(["delta_white_fraction", "id"])
        examples += [
            ("Largest white-saturation change / failure candidate", clipping.iloc[-1]["id"])
        ]
    skipped = analysis[analysis.decision == "skip"].sort_values("id")
    if len(skipped):
        examples.append(("Skipped control", skipped.iloc[0]["id"]))
    fig, axes = plt.subplots(len(examples), 2, figsize=(12, 3 * len(examples)), squeeze=False)
    for i, (label, sample_id) in enumerate(examples):
        for j, branch in enumerate(["raw", "adaptive"]):
            row = tables[branch].set_index("id").loc[sample_id]
            axes[i, j].imshow(cv2.cvtColor(read_image(row.image), cv2.COLOR_BGR2RGB))
            axes[i, j].set_title(f"{label} · {branch} · {sample_id}", fontsize=8)
            axes[i, j].axis("off")
    fig.suptitle("Representative visual checks · proxy ranking, not detector success/failure")
    fig.tight_layout()
    # Source-image examples stay gitignored; the notebook retains inline inspection evidence.
    fig.savefig(pair_root / "examples.png")
    plt.show()

Inspect adverse examples and distributions alongside averages. Proxy ranking is not detector success/failure, and sequential offline timing is not a deployment-speed claim.

<a id="development-synthetic-evidence"></a>

## 6. Development Inputs and Synthetic Evidence

Apply the already frozen recipe to original development sources without tuning. Synthetic calibration screens remain separate evidence and do not release holdout.

### 6.1 Matched Development Inputs

In [ ]:
if not FIVE_CLASS_ONLY:
    development = export_development_pair(
        protocol, policy, RAW_DEVELOPMENT, DEV_CONVERSION, RUN_ROOT / "development"
    )
    print("Verified development inputs:", development)

### 6.2 Synthetic Screening and the No-v3 Decision

In [ ]:
if not FIVE_CLASS_ONLY:
    from IPython.display import Image

    decision = read_json(ROOT / "docs/issue11-v2/v3_decision.json")
    assert decision["status"] == "v3_not_deployed"
    assert decision["retained_policy_id"] == policy["policy_id"]
    display(pd.read_csv(ROOT / "docs/issue11-v2/experimental_matrix.csv").head(12))
    display(Image(filename=str(ROOT / "docs/issue11-v2/supported_recovery.png"), width=900))
    print(decision["reason"])

The selected synthetic operator finding did not validate an independent router. Retain the exact Gamma-or-Skip baseline; archived snapshots contain full screening provenance.

<a id="export-integrity"></a>

## 7. Export and Integrity Summary

Write compact support/quality/cost evidence and identities after all exports. All six labels, paired invariants and untouched final-test pixels remain explicit.

In [ ]:
if not FIVE_CLASS_ONLY:
    subset_ids = {name: set(s["ids"]) for name, s in subsets["subsets"].items()}
    membership = pd.DataFrame(
        [
            {
                "id": r["id"],
                "group": r["group"],
                **{name: r["id"] in subset_ids[name] for name, s in subsets["subsets"].items()},
            }
            for r in sorted(pool, key=lambda r: r["id"])
        ]
    )
    member_path = REPORT_ROOT / "subset_membership.csv.gz"
    compressed = gzip.compress(membership.to_csv(index=False).encode(), mtime=0)
    if member_path.exists() and member_path.read_bytes() != compressed:
        raise ValueError("Committed subset membership changed; choose a new version/root")
    member_path.write_bytes(compressed)
    write_json(REPORT_ROOT / "policy.json", policy)
    summary = {
        "protocol_id": protocol["manifest_id"],
        "subsets_id": subsets["manifest_id"],
        "policy_id": policy["policy_id"],
        "policy_version": policy["version"],
        "route_support": policy["route_support"],
        "noise_diagnostics": policy["noise_diagnostics"],
        "membership_sha256": file_hash(member_path),
        "subsets": {
            name: {k: v for k, v in s.items() if k not in {"ids", "groups"}}
            for name, s in subsets["subsets"].items()
        },
        "materialized_subset": EXPORT_SUBSET,
        "images_per_branch": len(selected),
        "pair": read_json(pair_root / "pair.json"),
        "exports": {
            b: {
                "identity": read_json(pair_root / b / "export.json"),
                "verified": read_json(pair_root / b / "verified.json"),
            }
            for b in tables
        },
        "skip_rate": float((analysis.decision == "skip").mean()),
        "decisions": {k: int(v) for k, v in analysis.decision.value_counts().items()},
        "proxy_improved": int((analysis.proxy_loss_delta < -1e-8).sum()),
        "proxy_worsened": int((analysis.proxy_loss_delta > 1e-8).sum()),
        "mean_proxy_loss_delta": float(analysis.proxy_loss_delta.mean()),
        "examples": examples,
        "offline_latency": latency.to_dict("records"),
        "actual_image_GiB": {
            b: sum(Path(p).stat().st_size for p in t.image) / 2**30 for b, t in tables.items()
        },
        "environment": {
            "python": platform.python_version(),
            "opencv": cv2.__version__,
            "numpy": np.__version__,
            "cpu_count": os.cpu_count(),
            "opencv_threads": cv2.getNumThreads(),
            "machine": platform.machine(),
        },
        "final_test_pixels_accessed": False,
        "development_pixels_accessed": True,
        "detector_run": False,
        "training_run": False,
        "support_audit_sha256": file_hash(REPORT_ROOT / "subset_support.csv"),
        "paired_quality_sha256": file_hash(pair_root / "paired_quality.csv"),
        "limitations": policy["limits"] + subsets["limitations"],
    }
    write_json(REPORT_ROOT / "summary.json", summary)
    print(
        "PASS: nested immutable subsets; calibration-only frozen recipe; paired JPEG95 S10 exports"
    )
    display(
        pd.Series(
            {
                "S10": len(subsets["subsets"]["S10"]["ids"]),
                "S20": len(subsets["subsets"]["S20"]["ids"]),
                "Raw GiB": summary["actual_image_GiB"]["raw"],
                "Adaptive GiB": summary["actual_image_GiB"]["adaptive"],
            }
        )
    )

<a id="five-class-preparation"></a>

## 8. Derived Five-Class Inputs and Summary

Keep the original six-class conversion and exports untouched. Derive only a **new** class mapping from their frozen YOLO labels, preserving all frame IDs, surviving box coordinates and negative frames. Materialize five-class S10 and development Raw/Adaptive exports under `runs/preprocessing-five-v1/`; the final holdout remains sealed.


In [ ]:
from vehicle_detection.datasets import verify_role
from vehicle_detection.five_class import FIVE_CLASSES, derive_labels, support_audit
from vehicle_detection.offline import export_variant

if FIVE_CLASS_ONLY:
    subsets = validate_subsets(read_json(FROZEN_SUBSETS), protocol)
    policy = validate_policy(read_json(FROZEN_POLICY), protocol)
    pool = protocol["splits"]["training_pool"]
    records, _, sources = scan_bdd(DATA_ROOT, strict=False)
    for split, source in sources.items():
        assert source["sha256"] == protocol["annotation_sources"][split]["sha256"]
    partitions = training_records(protocol, subsets, EXPORT_SUBSET, records, DATA_ROOT)
    converted = TRAIN_CONVERSION
selected_ids = set(subsets["subsets"][EXPORT_SUBSET]["ids"])
train_rows = [r for r in pool if r["id"] in selected_ids]
dev_rows = protocol["splits"]["development_validation"]
dev_sources = [
    {"id": r["id"], "path": str(DATA_ROOT / r["source_path"]), "sha256": r["sha256"]}
    for r in dev_rows
]
FIVE_ROOT.mkdir(parents=True, exist_ok=True)
sample = [
    partitions["train"][i]
    for i in np.linspace(0, len(train_rows) - 1, min(16, len(train_rows)), dtype=int)
] + [dev_sources[i] for i in np.linspace(0, len(dev_rows) - 1, min(16, len(dev_rows)), dtype=int)]
budget = estimate_disk(
    sample, len(train_rows) + len(dev_rows), ["raw_reencoded", "adaptive"], policy["recipes"]
)
required = float(budget.p90_budget_GiB.sum() * 1.25 + 1)
free = shutil.disk_usage(FIVE_ROOT).free / 2**30
display(budget)
assert free >= required, "Insufficient estimated storage; do not start the export"
train_conversion = FIVE_ROOT / "conversion" / EXPORT_SUBSET
train_identity = derive_labels(converted, train_conversion, {"train": train_rows})
five_pair = FIVE_ROOT / "datasets" / EXPORT_SUBSET
five_tables = export_pair(
    partitions,
    train_conversion,
    five_pair,
    policy,
    protocol,
    subsets,
    EXPORT_SUBSET,
)
for branch, variant in [("raw", "raw_reencoded"), ("adaptive", "adaptive")]:
    verify_role(
        five_pair / branch,
        train_rows,
        "train",
        variant,
        policy["recipes"] if branch == "adaptive" else None,
    )
assert verify_alignment([five_pair / branch for branch in five_tables], deep=False) == len(
    train_rows
)

dev_conversion = FIVE_ROOT / "conversion" / "development"
dev_identity = derive_labels(DEV_CONVERSION, dev_conversion, {"tune": dev_rows})
five_dev_root = FIVE_ROOT / "development"
five_raw_dev = five_dev_root / "raw"
export_variant(
    {"tune": dev_sources},
    dev_conversion,
    five_raw_dev,
    "raw_reencoded",
    policy["recipes"],
)
verify_role(
    five_raw_dev,
    dev_rows,
    "tune",
    "raw_reencoded",
)
five_development = export_development_pair(
    protocol,
    policy,
    five_raw_dev,
    dev_conversion,
    five_dev_root,
)
role_support = {role: support_audit(rows) for role, rows in protocol["splits"].items()}
subset_support = {}
for name, subset in subsets["subsets"].items():
    ids = set(subset["ids"])
    subset_support[name] = support_audit([r for r in pool if r["id"] in ids])
display(pd.DataFrame([{"role": role, **values} for role, values in role_support.items()]).fillna(0))
write_json(
    FIVE_ROOT / "summary.json",
    {
        "source_protocol_id": protocol["manifest_id"],
        "source_subset_id": subsets["manifest_id"],
        "taxonomy": FIVE_CLASSES,
        "role_support_annotation_only": role_support,
        "subset_support_annotation_only": subset_support,
        "railway_exclusion_reason": "Insufficient support; not a measured AP improvement.",
        "storage_estimate": budget.to_dict("records"),
        "actual_image_GiB": {
            str(path.relative_to(FIVE_ROOT)): sum(
                Path(p).stat().st_size for p in pd.read_csv(path / "metadata.csv").image
            )
            / 2**30
            for path in [five_pair / b for b in five_tables] + list(five_development.values())
        },
        "training_conversion": train_identity,
        "development_conversion": dev_identity,
        "training_pair": read_json(five_pair / "pair.json"),
        "development_pair": read_json(five_dev_root / "pair.json"),
        "final_test_pixels_accessed": False,
    },
)
print("Five-class exports:", five_pair, five_development)
print("Original six-class exports and reserved final test remain untouched.")